# 7 — Add a Composite factory

> **Lesson focus**
>
> **Learn:** package repeated private topology behind one catalog
> factory. **Run:** define a grounded parallel-LC `CompositePlan`.
> **Inspect:** the resulting component’s public parameters and pin.
> **Status:** `CONVERGING` scaffold.

## Build the topology once inside the factory

Rebuilding the same parallel capacitor and inductor in every outer Plan
is noisy and exposes internals that consumers should not wire directly.
The custom `Library` from lesson 6 therefore gains one instance method.

Inside that method, `CompositePlan` owns the private construction graph.
`ParameterSpec` declares the two public physical parameters, while the
built-in `scnsim.components` catalog creates the primitive children.
Internal nodes and ground stay hidden until `expose_pin()` deliberately
publishes the one supported terminal. `build()` returns an immutable
`ComponentInstance`.

The adjacent source below is the one canonical catalog implementation;
the Tutorial includes it directly instead of maintaining a copied code
sample.

In [ ]:
"""One custom component catalog used by lessons 7 and 8."""

from __future__ import annotations

from scnsim import (
    ComponentInstance,
    CompositePlan,
    Library,
    ParameterSpec,
    components as builtin_components,
    units as u,
)


class ResonatorLibrary(Library):
    """Project catalog containing only reusable resonator factories."""

    def parallel_linear_lc_resonator(
        self,
        *,
        id: str,
        capacitance: object,
        inductance: object,
    ) -> ComponentInstance:
        """Build a grounded parallel LC with one public terminal."""

        composite = CompositePlan(id=id, library=self)
        capacitance_ref = composite.parameter(
            id="capacitance",
            baseline=capacitance,
            spec=ParameterSpec(unit=u.fF),
        )
        inductance_ref = composite.parameter(
            id="inductance",
            baseline=inductance,
            spec=ParameterSpec(unit=u.nH),
        )
        capacitor = composite.add(
            builtin_components.capacitor(
                id="capacitor",
                capacitance=capacitance_ref,
            )
        )
        inductor = composite.add(
            builtin_components.inductor(
                id="inductor",
                inductance=inductance_ref,
            )
        )
        terminal = composite.net(
            capacitor.pin("terminal_1"),
            inductor.pin("terminal_1"),
        )
        composite.ground(
            capacitor.pin("terminal_2"),
            inductor.pin("terminal_2"),
        )
        composite.expose_pin(id="terminal", at=terminal)
        return composite.build()


components = ResonatorLibrary()
"""Immutable custom component catalog exported by this module."""

## Create the component through its public surface

The module exports one singleton named `components`. Its custom factory
exposes only the two declared values and the terminal; the capacitor,
inductor, internal node, and ground group remain implementation
evidence.

In [ ]:
resonator = components.parallel_linear_lc_resonator(
    id="resonator",
    capacitance=110.0 * u.fF,
    inductance=5.8 * u.nH,
)
resonator.parameter("capacitance").show()
resonator.pin("terminal")

[Previous](06_create_library.qmd) · [Course map](../../docs/index.qmd) ·
[Next: use the custom component](08_use_custom_component.qmd) ·
[Concept: reusable
composition](../../docs/concepts/physical-authority-and-reusable-composition.qmd#composite-as-reuse-boundary)